# Neural Style Transfer with AdaIN

Transfer the artistic style of any painting onto a content image using **Adaptive Instance Normalization (AdaIN)**.

**Tech Stack:** Python · PyTorch · VGG19 · Flask (for web app)

---

## How AdaIN Works (Interview Answer)

1. **Encode**: Pass both content and style images through a frozen VGG19 encoder to extract feature maps.
2. **AdaIN**: Normalize the content features (zero mean, unit variance), then shift them using the **mean and standard deviation of the style features**. This transfers the style's "statistics" onto the content's "structure".
3. **Decode**: Pass the AdaIN output through a learned decoder to reconstruct the stylized image.

```
Content Image ──► VGG Encoder ──► content_feat ──►
                                                    AdaIN(content_feat, style_feat) ──► Decoder ──► Output
Style Image   ──► VGG Encoder ──► style_feat   ──►
```

Only the **decoder is trained**. The VGG encoder is frozen (pretrained on ImageNet).

In [ ]:
# Run this cell only on Kaggle / Colab if packages are missing
# !pip install torch torchvision tqdm pillow

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import os
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

## Configuration
Set your paths here. On Kaggle, upload `vgg_normalised.pth` and `decoder_final.pth` as a dataset.

In [ ]:
# ── KAGGLE PATHS ──────────────────────────────────────────────────────────────
# If running on Kaggle, upload your weights as a dataset and set these:
# VGG_PATH     = '/kaggle/input/nst-weights/vgg_normalised.pth'
# DECODER_PATH = '/kaggle/input/nst-weights/decoder_final.pth'
# CONTENT_DIR  = '/kaggle/input/coco-2017/train2017'
# STYLE_DIR    = '/kaggle/input/wikiart/wikiart'

# ── LOCAL PATHS (default) ────────────────────────────────────────────────────
VGG_PATH     = 'NST_Code/vgg_normalised.pth'
DECODER_PATH = 'NST_Code/experiment/final_exp/decoder_final.pth'
CONTENT_DIR  = 'NST_Code/content_data'
STYLE_DIR    = 'NST_Code/style_data'
SAVE_DIR     = 'experiment/kaggle_run'

# Training hyperparameters
EPOCHS       = 1
BATCH_SIZE   = 4
LR           = 1e-4
STYLE_WEIGHT = 5.0
IMAGE_SIZE   = 512
CROP_SIZE    = 256

os.makedirs(SAVE_DIR, exist_ok=True)

## Download VGG Weights (if not available locally)
The VGG19 encoder weights (~54MB) are needed. Run the cell below if you don't have them.

In [ ]:
# Uncomment to download vgg_normalised.pth
# import urllib.request
# print('Downloading VGG weights...')
# urllib.request.urlretrieve(
#     'https://github.com/naoto0804/pytorch-AdaIN/raw/master/models/vgg_normalised.pth',
#     VGG_PATH
# )
# print('Done.')

---
## Step 1: AdaIN Core Functions

These two functions are the heart of the project.

In [ ]:
def calc_mean_std(feat, eps=1e-5):
    """Compute per-channel mean and std of a feature map. Shape: [B, C, H, W]"""
    B, C = feat.size()[:2]
    feat_mean = feat.view(B, C, -1).mean(dim=2).view(B, C, 1, 1)
    feat_var  = feat.view(B, C, -1).var(dim=2, unbiased=False) + eps
    feat_std  = feat_var.sqrt().view(B, C, 1, 1)
    return feat_mean, feat_std


def adain(content_feat, style_feat):
    """
    Adaptive Instance Normalization:
    - Normalize content features to zero mean / unit variance
    - Re-scale using style's mean and std
    """
    size = content_feat.size()
    style_mean, style_std   = calc_mean_std(style_feat)
    content_mean, content_std = calc_mean_std(content_feat)

    normalized = (content_feat - content_mean.expand(size)) / content_std.expand(size)
    return normalized * style_std.expand(size) + style_mean.expand(size)

---
## Step 2: Model Architecture

### VGGEncoder
Frozen VGG19 (pretrained). We only use layers up to `relu4_1` (layer index 31).  
We extract features at 4 intermediate layers for the style loss.

In [ ]:
class VGGEncoder(nn.Module):
    def __init__(self, vgg_path):
        super().__init__()
        vgg = nn.Sequential(
            nn.Conv2d(3, 3, (1, 1)),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(3, 64, (3, 3)),
            nn.ReLU(),   # relu1_1
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(64, 64, (3, 3)),
            nn.ReLU(),   # relu1_2
            nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(64, 128, (3, 3)),
            nn.ReLU(),   # relu2_1
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(128, 128, (3, 3)),
            nn.ReLU(),   # relu2_2
            nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(128, 256, (3, 3)),
            nn.ReLU(),   # relu3_1
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, (3, 3)),
            nn.ReLU(),
            nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 512, (3, 3)),
            nn.ReLU(),   # relu4_1  ← we stop here
        )
        vgg.load_state_dict(torch.load(vgg_path, map_location='cpu', weights_only=True))

        layers = list(vgg.children())
        self.enc_1 = nn.Sequential(*layers[:4])    # up to relu1_1
        self.enc_2 = nn.Sequential(*layers[4:11])  # up to relu2_1
        self.enc_3 = nn.Sequential(*layers[11:18]) # up to relu3_1
        self.enc_4 = nn.Sequential(*layers[18:])   # up to relu4_1

        # Freeze all encoder weights — we never train these
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, x, return_last_only=False):
        h1 = self.enc_1(x)
        h2 = self.enc_2(h1)
        h3 = self.enc_3(h2)
        h4 = self.enc_4(h3)
        if return_last_only:
            return h4
        return h1, h2, h3, h4

### Decoder
Mirrors the encoder architecture in reverse. This is the only part we **train**.  
Input: 512-channel AdaIN output → Output: 3-channel RGB image.

In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(512, 256, (3, 3)),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 128, (3, 3)),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(128, 128, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(128, 64, (3, 3)),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(64, 64, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(64, 3, (3, 3)),
        )

    def forward(self, x):
        return self.net(x)

---
## Step 3: Dataset & Transforms

In [ ]:
class ImageFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.files = [
            f for f in os.listdir(root)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root, self.files[idx])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img


def get_transform(resize, crop_size):
    return transforms.Compose([
        transforms.Resize(resize),
        transforms.RandomCrop(crop_size),
        transforms.ToTensor(),
    ])


# Quick check
if os.path.exists(CONTENT_DIR):
    content_ds = ImageFolderDataset(CONTENT_DIR, get_transform(IMAGE_SIZE, CROP_SIZE))
    style_ds   = ImageFolderDataset(STYLE_DIR,   get_transform(IMAGE_SIZE, CROP_SIZE))
    print(f'Content images: {len(content_ds)}')
    print(f'Style images:   {len(style_ds)}')
else:
    print('Dataset directories not found — update CONTENT_DIR and STYLE_DIR in the config cell.')

---
## Step 4: Training

### Loss Functions

**Content loss** — The decoder output's features (after re-encoding) should match the AdaIN target `t`:  
`L_content = MSE(encoder(output)[-1], t)`

**Style loss** — At every encoder layer, the mean and std of the output should match the style image:  
`L_style = Σ MSE(mean(g_feat), mean(s_feat)) + MSE(std(g_feat), std(s_feat))`

In [ ]:
def train(content_dir, style_dir, vgg_path, save_dir,
          epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, style_weight=STYLE_WEIGHT):

    encoder = VGGEncoder(vgg_path).to(DEVICE)
    decoder = Decoder().to(DEVICE)
    optimizer = optim.Adam(decoder.parameters(), lr=lr)
    mse = nn.MSELoss()

    transform = get_transform(IMAGE_SIZE, CROP_SIZE)
    content_loader = DataLoader(
        ImageFolderDataset(content_dir, transform),
        batch_size=batch_size, shuffle=True, drop_last=True
    )
    style_loader = DataLoader(
        ImageFolderDataset(style_dir, transform),
        batch_size=batch_size, shuffle=True, drop_last=True
    )

    encoder.eval()

    for epoch in range(epochs):
        loop = tqdm(zip(content_loader, style_loader),
                    total=min(len(content_loader), len(style_loader)),
                    desc=f'Epoch {epoch+1}/{epochs}')

        for content_batch, style_batch in loop:
            content_batch = content_batch.to(DEVICE)
            style_batch   = style_batch.to(DEVICE)

            # Encode both images
            c_feats = encoder(content_batch)   # (h1, h2, h3, h4)
            s_feats = encoder(style_batch)

            # AdaIN: transfer style statistics onto content features
            t = adain(c_feats[-1], s_feats[-1])

            # Decode to get stylized image
            output = decoder(t)

            # Re-encode the output to compute losses
            g_feats = encoder(output)

            # Content loss: output features should match AdaIN target t
            loss_c = mse(g_feats[-1], t)

            # Style loss: output statistics should match style at all layers
            loss_s = 0
            for g_f, s_f in zip(g_feats, s_feats):
                g_mean, g_std = calc_mean_std(g_f)
                s_mean, s_std = calc_mean_std(s_f)
                loss_s += mse(g_mean, s_mean) + mse(g_std, s_std)

            loss = loss_c + style_weight * loss_s

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loop.set_postfix(loss=f'{loss.item():.4f}',
                             c_loss=f'{loss_c.item():.4f}',
                             s_loss=f'{loss_s.item():.4f}')

        # Save checkpoint after each epoch
        torch.save(decoder.state_dict(), os.path.join(save_dir, f'decoder_epoch{epoch+1}.pth'))
        print(f'Checkpoint saved → {save_dir}/decoder_epoch{epoch+1}.pth')

    return encoder, decoder

In [ ]:
# ── Run Training ──────────────────────────────────────────────────────────────
# Skip this if you already have decoder_final.pth (use the inference section below instead)

RUN_TRAINING = False  # Set to True to train from scratch

if RUN_TRAINING:
    encoder, decoder = train(
        content_dir=CONTENT_DIR,
        style_dir=STYLE_DIR,
        vgg_path=VGG_PATH,
        save_dir=SAVE_DIR,
    )
    print('Training complete.')

---
## Step 5: Inference (Load Pre-trained Model)

The pre-trained decoder (`decoder_final.pth`) is already trained. Load it here to run style transfer without training.

In [ ]:
# Load encoder and pre-trained decoder
encoder = VGGEncoder(VGG_PATH).to(DEVICE)
decoder = Decoder().to(DEVICE)
decoder.load_state_dict(torch.load(DECODER_PATH, map_location=DEVICE, weights_only=True))

encoder.eval()
decoder.eval()

print('Models loaded successfully.')

In [ ]:
def load_image(path, size=512):
    transform = transforms.Compose([
        transforms.Resize(size),
        transforms.ToTensor(),
    ])
    img = Image.open(path).convert('RGB')
    return transform(img).unsqueeze(0).to(DEVICE)


def stylize(content_path, style_path, alpha=1.0):
    """
    alpha: style strength (0.0 = original content, 1.0 = full style transfer)
    """
    content_tensor = load_image(content_path)
    style_tensor   = load_image(style_path)

    with torch.no_grad():
        content_feat = encoder(content_tensor, return_last_only=True)
        style_feat   = encoder(style_tensor,   return_last_only=True)

        # AdaIN
        t = adain(content_feat, style_feat)

        # alpha blends between pure content and full style transfer
        t = alpha * t + (1 - alpha) * content_feat

        output = decoder(t)

    return output.squeeze(0).clamp(0, 1).cpu()


print('stylize() ready.')

---
## Step 6: Run Style Transfer & Visualize

In [ ]:
# ── Set your input images ─────────────────────────────────────────────────────
CONTENT_IMAGE = 'NST_Code/examples/brad_pitt.jpg'
STYLE_IMAGE   = 'NST_Code/examples/sketch.png'
ALPHA         = 1.0   # 0.0–1.0: style strength

# Run style transfer
output_tensor = stylize(CONTENT_IMAGE, STYLE_IMAGE, alpha=ALPHA)

# Save output
output_path = os.path.join(SAVE_DIR, 'output.jpg')
transforms.ToPILImage()(output_tensor).save(output_path)
print(f'Output saved → {output_path}')

In [ ]:
def show_results(content_path, style_path, output_tensor):
    content = Image.open(content_path).convert('RGB')
    style   = Image.open(style_path).convert('RGB')
    output  = transforms.ToPILImage()(output_tensor)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    titles = ['Content Image', 'Style Image', 'Stylized Output']
    images = [content, style, output]

    for ax, title, img in zip(axes, titles, images):
        ax.imshow(img)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f'Neural Style Transfer (AdaIN) — Style Strength: {ALPHA}',
                 fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'comparison.jpg'), bbox_inches='tight', dpi=150)
    plt.show()


show_results(CONTENT_IMAGE, STYLE_IMAGE, output_tensor)

---
## Try Different Style Strengths

In [ ]:
alphas = [0.25, 0.5, 0.75, 1.0]

fig, axes = plt.subplots(1, len(alphas), figsize=(20, 5))

for ax, alpha in zip(axes, alphas):
    out = stylize(CONTENT_IMAGE, STYLE_IMAGE, alpha=alpha)
    ax.imshow(transforms.ToPILImage()(out))
    ax.set_title(f'α = {alpha}', fontsize=13)
    ax.axis('off')

plt.suptitle('Effect of Style Strength (alpha)', fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'alpha_comparison.jpg'), bbox_inches='tight', dpi=150)
plt.show()

---
## Interview Quick Reference

| Question | Answer |
|---|---|
| What is AdaIN? | Normalizes content features then rescales with style's mean/std |
| Why freeze the encoder? | VGG is already trained on ImageNet — its features capture textures well; we only need to train the decoder |
| What does alpha do? | Blends between content features and AdaIN output — controls how much style is applied |
| Content loss? | MSE between re-encoded output and AdaIN target `t` (preserves structure) |
| Style loss? | MSE between per-channel mean/std of output vs style at 4 encoder layers (matches texture) |
| Why ReflectionPad instead of ZeroPad? | Avoids edge artifacts — reflects border pixels instead of padding with zeros |